In [ ]:
from collections import defaultdict
import csv

In [ ]:
def load_table(path):
    table = defaultdict(list) #element -> [(line, energy, intensity), ...]
    with open(path, newline="") as fh:
        for r in csv.DictReader(fh):
            table[r["element"]].append((r["line"], float(r["energy"]), float(r["intensity"])))
    return table

def weighted_energy(lines, prefix):
    """Intensity-weighted mean energy of all lines starting with 'Ka' or 'Kb'."""
    pts = [(E, I) for name, E, I in lines if name.startswith(prefix)]
    return sum(E * I for E, I in pts) / sum(I for _, I in pts)

def identify(r, table, top=3):
    """r = results[folder][key]. Compares the two fitted peaks with each element."""
    peaks = [r["peak1"], r["peak2"]]
    if None in peaks:
        print("Need two successful fits")
        return
    a, b = sorted(peaks, key=lambda p: p["mu"]) #lower energy = Kα, higher = Kβ
    sig_a, sig_b = a["fwhm"] / 2.3548, b["fwhm"] / 2.3548

    scores = []
    for el, lines in table.items():
        Ea, Eb = weighted_energy(lines, "Ka"), weighted_energy(lines, "Kb")
        chi2 = ((a["mu"] - Ea) / sig_a) ** 2 + ((b["mu"] - Eb) / sig_b) ** 2
        scores.append((chi2, el, Ea, Eb))

    print(f"measured: Kα {a['mu']:.2f} keV, Kβ {b['mu']:.2f} keV")
    for chi2, el, Ea, Eb in sorted(scores)[:top]:
        print(f"  {el:>3}: Kα {Ea:.2f}, Kβ {Eb:.2f}  (score {chi2:.2f})")

In [ ]:
table = load_table("xray_energies.csv")

for folder, d in results.items():
    if folder == "calibration":
        continue
    for key, r in d.items():
        print(f"\n{folder}/{key}")
        identify(r, table)